# Actividad 0 — Verificación del entorno

Este cuaderno comprueba que **los dos entornos** del curso estén bien instalados. No
depende de ningún otro archivo del repositorio: se puede abrir y correr tal cual.

Son dos partes y **cada una usa un kernel distinto**:

| Parte | Kernel | Qué revisa |
|---|---|---|
| 1 | `nwcst` | Librerías de datos y modelos, y **dos modelos de prueba**: una regresión lineal y una red LSTM |
| 2 | `geo` | GDAL con HDF5, y una descarga real de luces nocturnas sobre Bolivia |

Revisar que las librerías *importen* no alcanza: `pmdarima`, `torch` y `nowcast_lstm` se
compilan contra versiones específicas de `numpy` y fallan recién al **entrenar**. Por eso
la Parte 1 ajusta dos modelos de verdad.

---

## Instalación

Desde el **Anaconda Prompt**, parado en la carpeta donde están este cuaderno y los
dos archivos `environment*.yml`:

```
conda env create -f environment.yml
conda activate nwcst
```

Y, solo para quien vaya a trabajar luces nocturnas:

```
conda env create -f environment-geo.yml
conda activate geo
```

> **Si `conda env create` falla o se queda pegado**, está el archivo
> **`instalacion_manual.txt`**, en esta misma carpeta, con los mismos paquetes instalados
> uno por uno. Es la receta que se usó en los cursos anteriores y sirve para aislar cuál
> es el paquete que da problema.


---

## Parte 1 — Entorno de nowcasting (`nwcst`)

**Seleccionar el kernel `nwcst`** antes de correr las dos celdas que siguen.


In [ ]:
# ===========================================================================
# PARTE 1 - kernel `nwcst`
# ===========================================================================
import sys, platform
print("Python", sys.version.split()[0], "|", platform.system(), platform.machine())
print()

fallas = []

# (modulo que se importa, nombre del paquete, obligatorio)
modulos = [
    ("numpy",         "numpy",           True),
    ("pandas",        "pandas",          True),
    ("scipy",         "scipy",           True),
    ("dateutil",      "python-dateutil", True),
    ("sklearn",       "scikit-learn",    True),
    ("statsmodels",   "statsmodels",     True),
    ("pmdarima",      "pmdarima",        True),
    ("torch",         "pytorch",         True),
    ("nowcast_lstm",  "nowcast_lstm",    True),
    ("matplotlib",    "matplotlib",      True),
    ("seaborn",       "seaborn",         True),
    ("joblib",        "joblib",          True),
    ("tqdm",          "tqdm",            True),
    ("openpyxl",      "openpyxl",        True),   # Excel del BCB y del INE
    ("xlrd",          "xlrd",            True),   # .xls antiguos: serie Brent de la EIA
    ("requests",      "requests",        True),
    ("bs4",           "beautifulsoup4",  True),   # paginas del INE y del BCB
    ("lxml",          "lxml",            True),
    ("html5lib",      "html5lib",        False),
    ("eurostat",      "eurostat",        False),  # comercio UE-Bolivia
    ("trendspy",      "trendspy",        False),  # Google Trends (act4)
    ("shap",          "shap",            False),  # importancia de variables (act7)
    ("geopandas",     "geopandas",       False),
]

for mod, paquete, obligatorio in modulos:
    try:
        m = __import__(mod)
        v = getattr(m, "__version__", "instalado")
        print(f"  OK     {paquete:16} {v}")
    except Exception as e:
        marca = "FALLA " if obligatorio else "AVISO "
        print(f"  {marca} {paquete:16} no importa: {str(e).splitlines()[0][:44]}")
        if obligatorio:
            fallas.append(paquete)

print()
if fallas:
    print("FALTAN:", ", ".join(fallas))
    print("Ver instalacion_manual.txt para instalar esos paquetes sueltos.")
else:
    print("Todas las librerias obligatorias estan presentes.")


### Los dos modelos de prueba

Importar no es correr. Esta celda **entrena de verdad**: una regresión lineal de
`scikit-learn` sobre datos simulados, y una red LSTM de `nowcast_lstm` sobre un panel
trimestral simulado, que es exactamente la maquinaria de la sesión 6.

**Salida esperada:**

```
LinearRegression OK - R2 = 0.9407
LSTM OK - dimension de las predicciones: (41, 3)
```

Son 41 filas y no 44 porque `nowcast_lstm` necesita `n_timesteps` trimestres de historia
antes de poder predecir el primero.

La LSTM tarda entre 10 y 60 segundos. Es normal, y no necesita GPU.


In [ ]:
# --- Modelo 1: regresion lineal --------------------------------------------
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

rng = np.random.default_rng(42)
X_train = rng.standard_normal((100, 3))
y_train = X_train @ np.array([1.5, -2.0, 0.8]) + rng.standard_normal(100) * 0.5
X_test  = rng.standard_normal((20, 3))
y_test  = X_test @ np.array([1.5, -2.0, 0.8]) + rng.standard_normal(20) * 0.5

modelo = LinearRegression().fit(X_train, y_train)
r2 = r2_score(y_test, modelo.predict(X_test))

print(f"LinearRegression OK - R2 = {r2:.4f}")
print(f"Coeficientes: {modelo.coef_.round(3)}   (los verdaderos son [1.5, -2.0, 0.8])")


# --- Modelo 2: LSTM ---------------------------------------------------------
# Panel trimestral simulado: 40 trimestres de entrenamiento y 4 de prueba.
import torch
from nowcast_lstm.LSTM import LSTM

# Este curso corre entero en CPU. Si el entorno quedo con la compilacion de
# CUDA en vez de la de CPU, nowcast_lstm intenta cargar cuDNN y, si el DLL no
# esta, el KERNEL SE MUERE sin dar un error de Python que se pueda atrapar.
# Visto en esta maquina el 2026-09-20: "Invalid handle. Cannot load symbol
# cudnnGetVersion". Se apaga cuDNN, que el curso no usa para nada.
if torch.backends.cudnn.is_available():
    print("AVISO: este pytorch trae CUDA/cuDNN. Se apaga; el curso corre en CPU.")
    print("       Para dejar el entorno como pide environment.yml:")
    print("       conda install -n nwcst pytorch=*=cpu_mkl*")
    torch.backends.cudnn.enabled = False

rng = np.random.default_rng(0)
n_obs  = 44
fechas = pd.date_range("2010-01-01", periods=n_obs, freq="QS")
x1  = rng.standard_normal(n_obs)
x2  = rng.standard_normal(n_obs)
pib = 1.5 * x1 - 2.0 * x2 + rng.standard_normal(n_obs) * 0.3

df    = pd.DataFrame({"date": fechas, "pib": pib, "x1": x1, "x2": x2})
train = df.iloc[:40].reset_index(drop=True)
full  = df.reset_index(drop=True)

parametros = {
    "n_timesteps"            : 4,
    "fill_na_func"           : np.nanmean,
    "fill_ragged_edges_func" : np.nanmean,
    "n_models"               : 2,
    "train_episodes"         : 10,
    "batch_size"             : 30,
    "decay"                  : 0.98,
    "n_hidden"               : 8,
    "n_layers"               : 1,
    "dropout"                : 0.0,
    "criterion"              : torch.nn.MSELoss(),
    "optimizer"              : torch.optim.Adam,
    "optimizer_parameters"   : {"lr": 1e-2, "weight_decay": 0.0},
}

lstm = LSTM(data=train, target_variable="pib", **parametros)
lstm.train(quiet=True)
preds = lstm.predict(full)

print()
print(f"LSTM OK - dimension de las predicciones: {preds.shape}")
print(preds.tail(4).to_string(index=False))

print()
print("=" * 62)
print("PARTE 1 LISTA. Ahora cambiar el kernel a `geo` y seguir abajo.")
print("=" * 62)


---

## Parte 2 — Entorno GIS (`geo`)

**Cambiar el kernel a `geo`.** Las celdas de abajo no corren en `nwcst`.

Esta parte es **opcional**: solo la necesita quien vaya a procesar imágenes satelitales.

### La trampa que hay que conocer

Desde GDAL 3.11 el **driver de HDF5 va en un paquete aparte**, `libgdal-hdf5`. Sin ese
driver, `gdal.Open()` no lee ningún archivo VIIRS — y lo peor es que **no lanza ningún
error**: devuelve `None`, el cuaderno sigue corriendo y termina "bien" con una serie vacía.

Hay **dos** causas distintas, y la segunda es la frecuente:

1. **El paquete falta.** Se arregla con `conda install -n geo libgdal-hdf5`.
2. **El paquete está, pero GDAL no lo encuentra.** Conda define `GDAL_DRIVER_PATH` desde
   un *script de activación*. Un kernel de Jupyter normalmente se lanza llamando al
   `python.exe` del entorno **sin activarlo**, así que esa variable nunca se define y el
   plugin queda invisible aunque el `.dll` esté ahí. Verificado en esta máquina el
   2026-09-18: `libgdal-hdf5 3.11.4` instalado, `gdal_HDF5.dll` presente, y aun así GDAL
   no abría los archivos.

La celda de abajo arregla el caso 2 sola, **antes** de importar `osgeo`. Ese orden importa:
al revés, la comprobación falla aunque el entorno esté perfecto.


In [ ]:
# ===========================================================================
# PARTE 2 - kernel `geo`
# ===========================================================================
import os, sys
from pathlib import Path

def arreglar_entorno_gdal():
    """Apunta GDAL a sus carpetas de datos y de plugins dentro de ESTE entorno.

    Conda normalmente define GDAL_DATA y GDAL_DRIVER_PATH en un script de
    activacion, que un kernel de Jupyter casi nunca ejecuta. Sin esas variables
    el plugin gdal_HDF5.dll queda invisible. Es inofensivo si ya estaban puestas.
    """
    raiz = sys.prefix
    win  = os.path.join(raiz, "Library")
    base = win if os.path.isdir(win) else raiz          # Windows vs unix

    for var, sub in (("GDAL_DATA",        ("share", "gdal")),
                     ("GDAL_DRIVER_PATH", ("lib", "gdalplugins")),
                     ("PROJ_LIB",         ("share", "proj"))):
        ruta = os.path.join(base, *sub)
        if not os.environ.get(var) and os.path.isdir(ruta):
            os.environ[var] = ruta

    # El plugin se carga con un LoadLibrary comun, que busca en PATH.
    libbin = os.path.join(base, "bin")
    if os.path.isdir(libbin) and libbin not in os.environ.get("PATH", ""):
        os.environ["PATH"] = libbin + os.pathsep + os.environ.get("PATH", "")

arreglar_entorno_gdal()

import numpy as np

for mod, paquete in [("h5py", "h5py"), ("rasterio", "rasterio"), ("osgeo", "gdal"),
                     ("geopandas", "geopandas"), ("shapely", "shapely"),
                     ("pyproj", "pyproj"), ("fiona", "fiona"),
                     ("matplotlib", "matplotlib"), ("requests", "requests")]:
    try:
        m = __import__(mod)
        print(f"  OK     {paquete:12} {getattr(m, '__version__', 'instalado')}")
    except Exception as e:
        print(f"  FALLA  {paquete:12} {str(e).splitlines()[0][:50]}")

from osgeo import gdal
gdal.UseExceptions()
print()
print(f"  GDAL {gdal.__version__}")
print(f"  GDAL_DRIVER_PATH = {os.environ.get('GDAL_DRIVER_PATH', '(sin definir)')}")

if gdal.GetDriverByName("HDF5") is None:
    print("  FALLA  driver HDF5 ausente: cada tile VIIRS se leeria VACIO sin dar error.")
    print("         conda install -n geo libgdal-hdf5     (y reiniciar el kernel)")
else:
    print("  OK     driver HDF5 disponible")


### Descarga de prueba: un día completo sobre Bolivia

Hace falta un **token de NASA LAADS**, gratuito:

1. Crear cuenta en `https://ladsweb.modaps.eosdis.nasa.gov`
2. *My Account* → *Generate Token*
3. Pegarlo abajo, entre las comillas. **El token vence a los 60 días**: si la descarga
   falla con un error de autorización, casi siempre es eso.

La prueba baja **un solo día, pero los cuatro mosaicos** que hacen falta para cubrir
Bolivia entera:

| mosaico | qué cubre | % del territorio |
|---|---|---:|
| `h11v10` | La Paz, Cochabamba y casi toda Santa Cruz | 77.8 |
| `h11v11` | Potosí, Tarija, Chuquisaca | 14.8 |
| `h12v10` | la Chiquitanía y Puerto Suárez | 7.1 |
| `h11v09` | una franja del norte de Pando | 0.3 |

Esa es exactamente la lista que usa `act3_ntl`, así que esta prueba deja el día ya
descargado y no hay que volver a bajarlo. Falta un quinto mosaico, `h12v11`, que toca
el extremo sureste del país: son 0.02% del territorio, chaco sin luces, y se deja fuera
a propósito para no bajar 25% más en cada corrida. En el mapa se nota como un hilo sin
pintar en la esquina sureste.

Cada archivo pesa unos **25 MB**: la prueba descarga alrededor de **100 MB**. Se guardan
en `hfiles/<mosaico>/`, que es donde `act3_ntl` los va a buscar.

> Un día cualquiera puede salir muy nublado. Si el mapa sale casi vacío, no es la
> instalación: cambiar `DIA` por otra fecha y repetir.


In [ ]:
# --- Pegar aqui el token de NASA LAADS -------------------------------------
NASA_TOKEN = "YOUR_TOKEN_HERE"

DIA    = "2026-06-15"                                   # un solo dia
TILES  = ["h11v10", "h11v11", "h12v10", "h11v09"]       # Bolivia entera
HFILES = Path("..") / "hfiles/"                          # donde act3_ntl los busca

# --- un dia de VNP46A2 sobre Bolivia, los cuatro mosaicos -------------------
import requests, h5py
from datetime import date

BASE = "https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP46A2"
HEAD = {"Authorization": f"Bearer {NASA_TOKEN}"}

d   = date.fromisoformat(DIA)
doy = d.timetuple().tm_yday

# Un solo listado del dia sirve para los cuatro mosaicos.
try:
    r = requests.get(f"{BASE}/{d.year}/{doy:03d}.json", headers=HEAD, timeout=120)
    r.raise_for_status()
    contenido = r.json().get("content", [])
    print(f"  OK     listado {DIA} (dia {doy}): {len(contenido)} archivos en el mundo")
except Exception as e:
    contenido = []
    print(f"  FALLA  no se pudo listar {DIA} ({type(e).__name__})")
    print("         Casi siempre es el token vencido o mal pegado.")

descargados = []
for tile in TILES:
    enlaces = [f["downloadsLink"] for f in contenido
               if tile in f["downloadsLink"] and f["downloadsLink"].endswith(".h5")]
    if not enlaces:
        print(f"  AVISO  {tile}: no hay archivo ese dia")
        continue

    carpeta = HFILES / tile
    carpeta.mkdir(parents=True, exist_ok=True)
    destino = carpeta / enlaces[0].split("/")[-1]

    if destino.exists() and destino.stat().st_size > 1_000_000:
        print(f"  OK     {tile}: ya estaba descargado ({destino.stat().st_size/1e6:.1f} MB)")
    else:
        try:
            parcial = destino.with_suffix(".part")
            with requests.get(enlaces[0], headers=HEAD, stream=True, timeout=600) as resp:
                resp.raise_for_status()
                with open(parcial, "wb") as fh:
                    for trozo in resp.iter_content(chunk_size=1 << 20):
                        fh.write(trozo)
            parcial.replace(destino)
            print(f"  OK     {tile}: descargado ({destino.stat().st_size/1e6:.1f} MB)")
        except Exception as e:
            print(f"  FALLA  {tile}: {type(e).__name__}: {str(e)[:60]}")
            continue
    descargados.append(destino)

# --- que los archivos se puedan LEER, no solo que esten ---------------------
print()
for p in descargados:
    with h5py.File(p, "r") as hf:
        print(f"  OK     h5py  {p.name[:44]}  claves={list(hf.keys())[:2]}")
    ds = gdal.Open(str(p), gdal.GA_ReadOnly)
    if ds is None:
        print("  FALLA  gdal.Open devolvio None -> falta libgdal-hdf5")
    else:
        print(f"  OK     gdal  {len(ds.GetSubDatasets())} subdatasets")
        ds = None

print()
print(f"  {len(descargados)} de {len(TILES)} mosaicos listos para el mapa.")


### Mosaico sobre Bolivia y mapa

In [ ]:
# --- mosaico de los cuatro tiles, recorte a Bolivia y mapa ------------------
import geopandas as gpd, rasterio, matplotlib.pyplot as plt
import colorcet as cc
from rasterio.merge import merge as rio_merge
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio",
         "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

if not descargados:
    print("No hay archivos que dibujar: revisar la celda anterior.")
else:
    tmp = HFILES / "_tmp_act0"
    tmp.mkdir(parents=True, exist_ok=True)

    # 1. cada .h5 -> un .tif georreferenciado con las esquinas de SU mosaico.
    #    Se pasa a Float32 aqui para que el recorte pueda usar NaN como vacio.
    tifs = []
    for ruta in descargados:
        capa = gdal.Open(str(ruta), gdal.GA_ReadOnly)
        # La capa 2 de VNP46A2 v002 es Gap_Filled_DNB_BRDF-Corrected_NTL, que
        # es la misma que usa act3_ntl. No es DNB_At_Sensor_Radiance: los
        # cuadernos del Caribe dicen eso en un comentario y esta equivocado.
        sub  = capa.GetSubDatasets()[2][0]
        r    = gdal.Open(sub, gdal.GA_ReadOnly)
        meta = r.GetMetadata_Dict()
        H, V = int(meta["HorizontalTileNumber"]), int(meta["VerticalTileNumber"])
        oeste, norte = (10 * H) - 180, 90 - (10 * V)
        este,  sur   = oeste + 10, norte - 10

        tile = ruta.name.split(".")[2]
        tif  = tmp / f"_{tile}.tif"
        gdal.Translate(str(tif), r, options=gdal.TranslateOptions(gdal.ParseCommandLine(
            f"-ot Float32 -a_srs EPSG:4326 -a_ullr {oeste} {norte} {este} {sur}")))
        r = None; capa = None
        tifs.append(tif)
        print(f"  OK     {tile}: recuadro {oeste},{sur} a {este},{norte}")

    # 2. pegar los cuatro en un solo raster
    abiertos = [rasterio.open(t) for t in tifs]
    mosaico, trans = rio_merge(abiertos)
    perfil = abiertos[0].profile
    for a in abiertos:
        a.close()
    perfil.update(height=mosaico.shape[1], width=mosaico.shape[2], transform=trans)
    mosaico_tif = tmp / "_mosaico.tif"
    with rasterio.open(mosaico_tif, "w", **perfil) as dst:
        dst.write(mosaico)
    print(f"  OK     mosaico de {len(tifs)} tiles: {mosaico.shape[2]} x {mosaico.shape[1]} pixeles")

    # 3. recortar con el contorno del pais
    gdf  = gpd.read_file("https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_BOL_shp.zip")
    pais = gdf[gdf.geometry.notnull()].dissolve().to_crs("EPSG:4326")
    formas = [mapping(g) for g in pais.geometry]
    with rasterio.open(mosaico_tif) as src:
        recorte, trans_r = rio_mask(src, formas, crop=True, nodata=np.nan, filled=True)

    nfilas, ncolumnas = recorte.shape[1], recorte.shape[2]
    lons = trans_r.c + (np.arange(ncolumnas) + 0.5) * trans_r.a
    lats = trans_r.f + (np.arange(nfilas)     + 0.5) * trans_r.e

    datos = recorte[0].astype(float)
    # VNP46A2 v002 rellena con -999.9; las colecciones viejas usaban 65535.
    datos[datos >= 65535] = np.nan
    datos[datos < 0]      = np.nan
    validos = datos[np.isfinite(datos)]
    print(f"  OK     recorte {datos.shape}, {validos.size:,} pixeles dentro de Bolivia")

    # 4. el mapa
    LON, LAT   = np.meshgrid(lons, lats)
    vmin, vmax = np.nanpercentile(validos, 2), np.nanpercentile(validos, 98)
    etiqueta   = f"{d.day} de {MESES[d.month - 1]} de {d.year}"

    fig, ax = plt.subplots(figsize=(10, 8))
    pcm = ax.pcolormesh(LON, LAT, datos, cmap=cc.cm.bmy,
                        vmin=vmin, vmax=vmax, shading="auto")
    try:
        import contextily as cx
        cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, crs="EPSG:4326")
    except Exception as e:
        print(f"  AVISO  sin mapa base ({type(e).__name__}); el resto del mapa igual sirve")
    pais.boundary.plot(ax=ax, edgecolor="white", linewidth=0.6)

    fig.colorbar(pcm, ax=ax, orientation="vertical",
                 label="Radiancia NTL (nW/cm2/sr)")
    ax.set_title(f"Luces nocturnas - Bolivia ({etiqueta})", fontsize=13, weight="bold")
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    fig.text(0.01, 0.01, "Fuente: NASA Black Marble VNP46A2", fontsize=9)
    plt.tight_layout()
    plt.show()

    # 5. los .tif eran intermedios: se borran, los .h5 se quedan para act3_ntl
    for t in tifs + [mosaico_tif]:
        try:
            t.unlink()
        except OSError:
            pass
    try:
        tmp.rmdir()
    except OSError:
        pass

    print()
    print("=" * 62)
    print("TODO LISTO. Los dos entornos funcionan.")
    print("=" * 62)


# Parte 3 - Configurar los paths de R en VS Code

Este notebook escribe las claves `r.rpath` / `r.rterm` en el `settings.json` de VS Code, para que la extensión de R encuentre tu instalación sin configurarla a mano.

**Cómo usarlo:** ejecuta la celda de abajo una vez (usa solo Python, el kernel de este notebook). Detecta R aunque no esté en el PATH, respeta lo que ya tengas en `settings.json` (deja un `.bak`), y funciona en Windows, macOS y Linux. Al terminar, **cierra VS Code por completo y vuelve a abrirlo**.

> Alternativa: si R ya está en el PATH del sistema, no necesitas esto; la extensión lo encuentra sola.

In [ ]:
# Configura los paths de R en el settings.json de VS Code
# Ejecuta esta celda UNA vez. Requiere solo Python (kernel de este notebook).
# Al terminar, CIERRA VS Code por completo y vuelve a abrirlo.

import json, os, re, glob, platform
from pathlib import Path

system = platform.system()  # 'Windows', 'Darwin', 'Linux'

# ---------- 1) Localizar los ejecutables de R ----------
def ver_key(p):
    m = re.search(r"R-([\d.]+)", str(p))
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)

def find_r():
    """Devuelve (rpath, rterm) con barras normales, o (None, None)."""
    if system == "Windows":
        bases = [r"C:\Program Files\R", r"C:\Program Files (x86)\R",
                 os.path.expandvars(r"%LOCALAPPDATA%\Programs\R")]
        bindirs = []
        for base in bases:
            bindirs += [Path(g).parent for g in glob.glob(os.path.join(base, "R-*", "bin", "x64", "R.exe"))]
            bindirs += [Path(g).parent for g in glob.glob(os.path.join(base, "R-*", "bin", "R.exe"))]
        for b in sorted(set(bindirs), key=ver_key, reverse=True):
            if (b / "R.exe").exists() and (b / "Rterm.exe").exists():
                return str(b / "R.exe").replace("\\", "/"), str(b / "Rterm.exe").replace("\\", "/")
        return None, None
    else:
        import shutil
        rbin = shutil.which("R")
        return (rbin, rbin) if rbin else (None, None)

rpath, rterm = find_r()
if not rpath:
    raise SystemExit(
        "No encontré R automáticamente. Abre R (RGui) y corre R.home('bin') "
        "para ver la ruta, luego agrega a mano en settings.json:\n"
        '  "r.rpath.windows": "<ruta>/R.exe",\n  "r.rterm.windows": "<ruta>/Rterm.exe"')
print("R encontrado:")
print("  rpath:", rpath)
print("  rterm:", rterm)

# ---------- 2) Localizar el settings.json del usuario ----------
if system == "Windows":
    settings = Path(os.environ["APPDATA"]) / "Code" / "User" / "settings.json"
elif system == "Darwin":
    settings = Path.home() / "Library" / "Application Support" / "Code" / "User" / "settings.json"
else:
    settings = Path.home() / ".config" / "Code" / "User" / "settings.json"
settings.parent.mkdir(parents=True, exist_ok=True)

# ---------- 3) Leer settings.json tolerando comentarios (JSONC) ----------
def strip_jsonc(text):
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.S)   # bloques /* */
    out = []
    for line in text.splitlines():                       # // fuera de strings
        in_str = esc = False; res = ""; i = 0
        while i < len(line):
            c = line[i]
            if esc: res += c; esc = False
            elif c == "\\" and in_str: res += c; esc = True
            elif c == '"': in_str = not in_str; res += c
            elif c == "/" and i + 1 < len(line) and line[i+1] == "/" and not in_str: break
            else: res += c
            i += 1
        out.append(res)
    return re.sub(r",(\s*[}\]])", r"\1", "\n".join(out))  # comas finales

data = {}
if settings.exists() and settings.stat().st_size > 0:
    raw = settings.read_text(encoding="utf-8")
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        try:
            data = json.loads(strip_jsonc(raw))
        except json.JSONDecodeError:
            raise SystemExit(
                f"No pude parsear {settings}. Agrega estas claves a mano:\n"
                f'  "r.rpath.{ "windows" if system=="Windows" else "mac" if system=="Darwin" else "linux" }": "{rpath}",\n'
                f'  "r.rterm.{ "windows" if system=="Windows" else "mac" if system=="Darwin" else "linux" }": "{rterm}"')

# ---------- 4) Fijar las claves de la plataforma correcta ----------
suf = "windows" if system == "Windows" else "mac" if system == "Darwin" else "linux"
data[f"r.rpath.{suf}"] = rpath
data[f"r.rterm.{suf}"] = rterm

# ---------- 5) Respaldo y escritura ----------
if settings.exists():
    settings.with_suffix(".json.bak").write_text(settings.read_text(encoding="utf-8"), encoding="utf-8")
settings.write_text(json.dumps(data, indent=4, ensure_ascii=False) + "\n", encoding="utf-8")

print("\nsettings.json actualizado:", settings)
print("Respaldo:", settings.with_suffix(".json.bak"))
print("\nListo. CIERRA VS Code por completo y vuelve a abrirlo para que tome los paths.")


R encontrado:
  rpath: C:/Program Files/R/R-4.6.1/bin/x64/R.exe
  rterm: C:/Program Files/R/R-4.6.1/bin/x64/Rterm.exe

settings.json actualizado: C:\Users\User\AppData\Roaming\Code\User\settings.json
Respaldo: C:\Users\User\AppData\Roaming\Code\User\settings.json.bak

Listo. CIERRA VS Code por completo y vuelve a abrirlo para que tome los paths.
